# Read the size-dependent fate and pair coefficients

**Role:** Analysis.

Load one saved exclusive-fate spline model at a selected radius, fold and inner-split seed. Plot its shared size curve b(N), center effects a(N), shared neighborhood sensitivities c(N), and ordered center–neighbor deviations P(N). Also show c+P, the total sensitivity for a selected center, and check that the displayed terms reconstruct predictions exactly. There is no ablation inference, training or re-estimation of reference compositions.

All curves are in physical curvature units. b+a describes the selected model at its fixed reference neighborhood, not a separately fitted depth-zero model or a cell's intrinsic curvature. c and P are centered coefficient functions; a positive P means more positive than the shared response, not necessarily positive total sensitivity. Fractions are compositional, so meaningful changes compare source identities. Pair effects need not be symmetric and do not establish causal signaling. Sparse support, anatomical correlations and size-as-pseudotime remain limitations.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/models/gnn.py').is_file())  # Repository root.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Resolve data and saved-run paths relative to this directory.

## Analysis settings

Select one model explicitly near the top. This avoids averaging folds with different training-derived reference conventions. Curves use the model's training N range; no unsupported extrapolation is plotted. Choose centers and sources to limit the figure count, then inspect training support for those same pairs. Uncertainty bands are deliberately absent: these are fitted coefficient functions, not confidence intervals.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.artifacts.paths import resolve_training_run
from src.artifacts.runs import AnalysisRun
from src.training.spline_fit import graph_samples
from src.models.fate_spline import FateSplineCurvature

# Saved model
TRAINING_RUN = None  # Name under training_results/fate_spline_training; None requires exactly one run.
MODEL_VARIANT = 'pairwise'  # center, shared or pairwise.
MODEL_RADIUS = 2  # Maximum shell radius; use 0 for center-only.
MODEL_FOLD = 0  # One saved outer fold; do not mix coefficient reference conventions.
MODEL_SEED = 42  # Saved inner-holdout seed.

# Coefficient plots
CENTER_MARKERS = ['LGR5', 'Serotonin', 'Lysozyme']  # Center identities shown for P and c+P; None shows all.
SOURCE_MARKERS = None  # Source identities shown; None includes every marker and Unassigned.
N_RANGE = None  # Optional (min, max) display interval within the model's training support.
N_POINTS = 150  # Number of log-spaced evaluation counts.
N_SUPPORT_BINS = 6  # Log-N bins for observed training pair support.
CONTRAST_SOURCE = 'Unassigned'  # Explicit comparison identity for source swaps; never an implicit neutral fate.

# Example decomposition
EXAMPLE_ORGANOID = None  # Validation organoid ID; None chooses the first containing the selected center.
EXAMPLE_CENTER = 'LGR5'  # Identity of the example prediction recipient.
EXAMPLE_NODE = None  # Optional exact node index; None selects the first matching center.

## Restore the constrained coefficient functions

The fitted model uses weighted sum-to-zero constraints: center effects average to zero over saved center weights; shared source coefficients average to zero over saved source weights; P averages to zero over both center and source weights. These references are fixed across N. The source composition is averaged over nonempty shells within organoid, then equally across organoids. An empty shell contributes zero. Save the reference weights beside the figures so a coefficient sign can always be interpreted.

In [ ]:
RUN_DIR = resolve_training_run(ROOT, 'fate_spline_training', TRAINING_RUN)
run = AnalysisRun(RUN_DIR)
rows = run.records[(run.records.name == f'spline_{MODEL_VARIANT}') & (run.records.depth == MODEL_RADIUS) &
                   (run.records.fold == MODEL_FOLD) & (run.records.seed == MODEL_SEED)]
if len(rows) != 1:
    raise ValueError('Select exactly one saved variant/radius/fold/seed; inspect models.json.')
selected = run.select(rows.iloc[0].key)
model = selected['model']
if not isinstance(model, FateSplineCurvature):
    raise TypeError('This notebook requires a fitted FateSplineCurvature model.')
identities = [*selected['marker_names'], 'Unassigned']
centers = identities if CENTER_MARKERS is None else CENTER_MARKERS
sources = identities if SOURCE_MARKERS is None else SOURCE_MARKERS
if not centers or not sources or not set([*centers, *sources, CONTRAST_SOURCE]) <= set(identities):
    raise ValueError(f'Choose identities from {identities}.')
lo, hi = np.exp(model.log_count_range.cpu().numpy())
if N_RANGE is not None:
    if len(N_RANGE) != 2 or not lo <= N_RANGE[0] < N_RANGE[1] <= hi:
        raise ValueError(f'N_RANGE must lie within training support [{lo:g}, {hi:g}].')
    lo, hi = N_RANGE
if N_POINTS < 2 or N_SUPPORT_BINS < 1:
    raise ValueError('Need >=2 N points and >=1 support bin.')
N = np.geomspace(lo, hi, N_POINTS)
coefs = model.coefficients(N)
OUTPUT_DIR = RUN_DIR / 'analysis' / 'coefficients' / rows.iloc[0].key
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
reference = dict(center_weights=dict(zip(identities, model.center_reference.tolist())),
    source_weights={str(r+1):dict(zip(identities, row)) for r,row in enumerate(model.source_reference.tolist())})
(OUTPUT_DIR / 'reference_weights.json').write_text(json.dumps(reference, indent=2))
(OUTPUT_DIR / 'settings.json').write_text(json.dumps(dict(model_key=rows.iloc[0].key, centers=centers, sources=sources,
    N_range=[float(lo),float(hi)], contrast_source=CONTRAST_SOURCE, units='physical curvature',
    residualized=selected['residualized'], interpretation='conditional predictive associations; no causal identification'), indent=2))
print('Loaded:', rows.iloc[0].key, '| training N range:', np.exp(model.log_count_range.cpu().numpy()))
print('Residualized:', selected['residualized'], '| terms exclude the saved organoid-specific baseline.')
print('Selected penalties:', selected['metrics']['best'])
colors = {name:plt.get_cmap('tab10')(i % 10) for i,name in enumerate(identities)}

## Center identity and the shared size curve

The left panel shows b(N) and b(N)+a_A(N) at the reference neighborhood. The right panel isolates a_A(N). With residualization, these are deviations from the saved baseline; the baseline is organoid-specific and is not part of these universal curves. Center coefficients are not claims about intrinsic cell mechanics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(N, coefs['b'], color='black', linestyle='--', label='b(N)')
for name in identities:
    a = coefs['a'][:, identities.index(name)]
    axes[0].plot(N, coefs['b']+a, color=colors[name], label=name)
    axes[1].plot(N, a, color=colors[name], label=name)
for ax, title in zip(axes, ['Center prediction at reference neighborhood: b+a', 'Center deviation: a']):
    ax.set(xscale='log', xlabel='Cell count N', ylabel='Curvature contribution', title=title)
    ax.axhline(0, color='gray', linewidth=.6)
    ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'center_terms.png', dpi=160)
plt.show()
center_table = pd.DataFrame({'N':N, 'b':coefs['b']})
for name in identities:
    center_table[f'a_{name}'] = coefs['a'][:,identities.index(name)]
center_table.to_csv(OUTPUT_DIR / 'center_terms.csv', index=False)

## Neighborhood responses: shared and center-dependent

First show c_B^r(N), the shared source sensitivity, by exact hop. Then show P_AB^r(N) and c_B^r(N)+P_AB^r(N) for selected centers. Each plotted coefficient is per unit composition, not per neighboring cell. Actual contributions multiply the coefficient by p_B−π_B; the observed neighborhood therefore matters even when the sensitivity is large. A positive P can coexist with a negative total response.

In [ ]:
if model.radius:
    fig, axes = plt.subplots(1, model.radius, figsize=(5*model.radius, 4), squeeze=False)
    for r, ax in enumerate(axes[0]):
        for source in sources:
            ax.plot(N, coefs['c'][:,r,identities.index(source)], color=colors[source], label=source)
        ax.set(xscale='log', xlabel='Cell count N', ylabel='c: curvature / unit fraction', title=f'Shared sensitivity — hop {r+1}')
        ax.axhline(0, color='gray', linewidth=.6)
        ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'shared_terms.png', dpi=160)
    plt.show()
    fig, axes = plt.subplots(len(centers), model.radius, figsize=(5*model.radius, 3.5*len(centers)), squeeze=False)
    coefficient_rows = []
    for center, row in zip(centers, axes):
        a = identities.index(center)
        for r, ax in enumerate(row):
            for source in sources:
                b = identities.index(source)
                p = coefs['P'][:,r,a,b]
                total = coefs['c'][:,r,b] + p
                ax.plot(N, total, color=colors[source], label=source)
                ax.plot(N, p, '--', color=colors[source], alpha=.75)
                coefficient_rows.extend(dict(N=float(n), center=center, source=source, hop=r+1,
                    shared=float(c), pairwise=float(v), total=float(t)) for n,c,v,t in zip(N,coefs['c'][:,r,b],p,total))
            ax.set(xscale='log', xlabel='Cell count N', ylabel='Curvature / unit fraction', title=f'Center {center} — hop {r+1}')
            ax.axhline(0, color='gray', linewidth=.6)
            ax.legend(fontsize=7)
    fig.suptitle('Solid: total sensitivity c+P; dashed: center-specific deviation P', y=1.01)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'pair_terms.png', dpi=160, bbox_inches='tight')
    plt.show()
    pd.DataFrame(coefficient_rows).to_csv(OUTPUT_DIR / 'neighborhood_coefficients.csv', index=False)

## Explicit identity contrasts and training support

Compare each source B to the named comparison identity C: (c_B−c_C)+(P_AB−P_AC). This is the exact change for replacing an entire unit fraction of C with B at fixed shell size; changing one neighbor divides it by that shell's cell count. C is a comparison choice, not a neutral or unknown fate. These contrasts are independent of how intercept/main-effect offsets are assigned.

Support panels count unique training organoids containing at least one selected center with at least one source in the indicated exact-hop shell. They do not count paths or treat neighboring cells as independent replicates. Smooth curves through sparse bins are regularized estimates; lack of support cannot be repaired by a spline. No confidence bands are inferred from the number of cells.

In [ ]:
training_samples = graph_samples(selected['groups']['train'], model.radius)
if model.radius:
    fig, axes = plt.subplots(len(centers), model.radius, figsize=(5*model.radius, 3.3*len(centers)), squeeze=False)
    cidx = identities.index(CONTRAST_SOURCE)
    for center, row in zip(centers, axes):
        a = identities.index(center)
        for r, ax in enumerate(row):
            for source in sources:
                b = identities.index(source)
                contrast = coefs['c'][:,r,b]-coefs['c'][:,r,cidx]+coefs['P'][:,r,a,b]-coefs['P'][:,r,a,cidx]
                ax.plot(N, contrast, color=colors[source], label=source)
            ax.axhline(0, color='gray', linewidth=.6)
            ax.set(xscale='log', xlabel='Cell count N', ylabel='Curvature / replaced fraction', title=f'{center}, hop {r+1}: B minus {CONTRAST_SOURCE}')
            ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'identity_contrasts.png', dpi=160)
    plt.show()
    support_rows = []
    for sample in training_samples:
        for center in centers:
            mask = sample['identity'] == identities.index(center)
            for r in range(model.radius):
                for source in sources:
                    count = int(np.sum(mask & (sample['counts'][:,r,identities.index(source)] > 0)))
                    support_rows.append(dict(organoid_str=sample['organoid_str'], N=sample['N'], center=center,
                                             source=source, hop=r+1, eligible_centers=count))
    support = pd.DataFrame(support_rows)
    edges = np.geomspace(min(s['N'] for s in training_samples)*.999999, max(s['N'] for s in training_samples)*1.000001, N_SUPPORT_BINS+1)
    support['N_bin'] = pd.cut(support.N, edges, labels=False, include_lowest=True)
    support.to_csv(OUTPUT_DIR / 'training_pair_support.csv', index=False)
    fig, axes = plt.subplots(len(centers), model.radius, figsize=(5*model.radius, 3*len(centers)), squeeze=False)
    for center, row in zip(centers, axes):
        for r, ax in enumerate(row):
            part = support[(support.center == center) & (support.hop == r+1) & (support.eligible_centers > 0)]
            table = part.groupby(['source','N_bin']).organoid_str.nunique().unstack(fill_value=0).reindex(index=sources, columns=range(N_SUPPORT_BINS), fill_value=0)
            image = ax.imshow(table, aspect='auto', cmap='Blues', vmin=0)
            ax.set(yticks=range(len(sources)), yticklabels=sources, xticks=range(N_SUPPORT_BINS),
                   xticklabels=[f'{edges[i]:.0f}–{edges[i+1]:.0f}' for i in range(N_SUPPORT_BINS)],
                   title=f'Training support: {center}, hop {r+1}', xlabel='Observed N bin')
            ax.tick_params(axis='x', rotation=45)
            fig.colorbar(image, ax=ax, label='Unique organoids')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'training_pair_support.png', dpi=160)
    plt.show()

## Check an individual prediction decomposition

Choose a validation recipient and display the organoid baseline offset, shared size term, center term, and actual shared/pair shell contributions. Their sum equals the model prediction in physical units. This uses observed composition and N, not a hypothetical no-neighbor cell. The validation truth is shown only for comparison; it did not enter fitting this checkpoint. In a nonresidualized run the baseline offset is zero even though a comparison baseline was saved.

In [ ]:
validation_samples = graph_samples(selected['groups']['val'], model.radius)
if EXAMPLE_CENTER not in identities:
    raise ValueError('EXAMPLE_CENTER is not in this model.')
candidates = [s for s in validation_samples if (EXAMPLE_ORGANOID is None or s['organoid_str'] == EXAMPLE_ORGANOID)
              and np.any(s['identity'] == identities.index(EXAMPLE_CENTER))]
if not candidates:
    raise ValueError('No matching validation organoid/center identity.')
example = candidates[0]
node = int(np.flatnonzero(example['identity'] == identities.index(EXAMPLE_CENTER))[0]) if EXAMPLE_NODE is None else EXAMPLE_NODE
if not 0 <= node < len(example['identity']) or example['identity'][node] != identities.index(EXAMPLE_CENTER):
    raise ValueError('EXAMPLE_NODE must have the selected center identity.')
terms = model.contributions(example)
np.testing.assert_allclose(terms['prediction'], model.predict_sample(example), rtol=1e-10, atol=1e-12)
offset = np.asarray(selected['baseline_offsets'][example['organoid_str']]).reshape(-1)[node]
values = {'Baseline offset':float(offset), 'Shared b(N)':float(terms['b'][node]), 'Center a(N)':float(terms['a'][node])}
for r in range(model.radius):
    values[f'Shared hop {r+1}'] = float(terms['shared'][node,r].sum())
    values[f'Pair hop {r+1}'] = float(terms['pairwise'][node,r].sum())
prediction = float(terms['prediction'][node] + offset)
truth = float(example['y'][node] + offset)
np.testing.assert_allclose(sum(values.values()), prediction, rtol=1e-10, atol=1e-12)
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(list(values), list(values.values()))
ax.axhline(0, color='gray', linewidth=.6)
ax.tick_params(axis='x', rotation=30)
ax.set(ylabel='Physical curvature contribution', title=f"{example['organoid_str']}, node {node}, N={example['N']}\nSum/prediction={prediction:.5g}; measured={truth:.5g}")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'prediction_decomposition.png', dpi=160)
plt.show()
pd.Series(values, name='contribution').to_csv(OUTPUT_DIR / 'example_decomposition.csv')